# Yvy PMTiles — GPKG → PMTiles (todo en sesión)

Genera los 7 PMTiles de `08_pmtiles/` desde `01_work/*.gpkg`. Todo se instala
en la sesión (nada cacheado): tippecanoe **2.79.0** compilado + geopandas.

**Contrato (SPEC §10b, no tocar):** cada feature lleva propiedad `id` numérica
(`--use-attribute-for-id=id`), un archivo por nivel/tipo, nombre = nivel o tipo.
El repo verifica el match en `pnpm run build` (avisa `X/Y con id coincidente`).

**Ids por nivel** (igual que `yvy_runner.celda_8`, con fix provincias):
`pais` = 0 (ya viene) · `provincias` = `1000+in1` · `departamentos` = `100000+in1` ·
`municipios` = `1000000+in1` · `barrios` = secuencia `1660063001+i` por `nom_largo` ·
`mineria`/`catastro` = índice interno. Filtro `geom_estado ∈ {ok, reparada}`.

In [ ]:
# 1) tippecanoe 2.79.0 desde source (pin SPEC) + geopandas + Drive
import subprocess
print(subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', '2.79.0',
                      'https://github.com/felt/tippecanoe.git', '/content/tippecanoe']).returncode)
!apt-get update -qq && apt-get install -y -qq build-essential libsqlite3-dev zlib1g-dev
!cd /content/tippecanoe && make -j$(nproc) && make install
!tippecanoe --version
%pip install -q geopandas pyogrio
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Config: GPKG -> capas -> PMTiles (zooms = headers actuales del repo)
from pathlib import Path
BASE = Path('/content/drive/MyDrive/YvyLabs/YvyExplorerProduction')
WORK, OUT = BASE / '01_work', BASE / '08_pmtiles'
OUT.mkdir(parents=True, exist_ok=True)
# Catálogo para join de barrios (subir data/registry/jurisdicciones.parquet a /content)
CAT_JURIS = Path('/content/jurisdicciones.parquet')

# (gpkg, capa, salida, zmin, zmax) — comentar niveles que NO se re-tilean
PLAN = [
    ('nacional.gpkg',   'pais',          'paises.pmtiles',        0, 8),
    ('provincial.gpkg', 'provincias',    'provincias.pmtiles',    4, 10),
    ('provincial.gpkg', 'departamentos', 'departamentos.pmtiles', 6, 12),
    ('municipal.gpkg',  'municipios',    'municipios.pmtiles',    8, 14),
    ('local.gpkg',      'barriosSalCap', 'barrios.pmtiles',       10, 16),
    ('provincial.gpkg', 'catMinero',     'mineria.pmtiles',       8, 14),
    ('provincial.gpkg', 'catastroUR',    'catastro.pmtiles',      12, 16),
]
import fiona
for gpkg, *_ in PLAN:
    p = WORK / gpkg
    print(('OK ' if p.exists() else 'FALTA ') + gpkg,
          fiona.listlayers(p) if p.exists() else '')

In [ ]:
# 2b) Preparar `pais`: 00_entrada/*.gpkg -> layer `pais` en 01_work/nacional.gpkg
# (solo si la celda 2 muestra nacional.gpkg sin capa pais)
import fiona, geopandas as gpd
ENTRADA = BASE / '00_entrada'
cands = [(p, fiona.listlayers(p)) for p in sorted(ENTRADA.glob('*.gpkg'))]
for p, layers in cands:
    print(p.name, '->', layers)
assert cands, 'No hay .gpkg en 00_entrada'
match = [(p, l) for p, layers in cands for l in layers
         if l.lower() in ('pais', 'argentina', 'limite', 'limites')]
if match:
    src, lyr = match[0]
elif len(cands) == 1 and len(cands[0][1]) == 1:
    src, lyr = cands[0][0], cands[0][1][0]
else:
    raise SystemExit('Varias capas y ninguna es pais: indicar manual')
print(f'fuente: {src.name} / capa: {lyr}')
gdf = gpd.read_file(src, layer=lyr)
assert len(gdf) == 1, f'pais debe ser 1 feature, hay {len(gdf)}'
gdf = gdf[['geometry']].copy()
gdf['yvy_id'] = 0
dest = WORK / 'nacional.gpkg'
gdf.to_file(dest, layer='pais', driver='GPKG')
print(f'OK layer pais en {dest} (crs={gdf.crs})')

In [ ]:
# 3) Tileado (réplica de tools/yvy_plantilla.py::gpkg_capa_a_pmtiles +
#    yvy_runner.celda_8, con fix provincias 1000+in1). Log en vivo.
import geopandas as gpd, subprocess

def asignar_id(gdf, capa, nombre):
    if nombre == 'provincias':
        gdf['id'] = gdf['in1'].astype(str).str.strip().apply(lambda x: 1000 + int(x))
        assert set(gdf['id']) == set(range(1002, 1095, 4)), 'in1 provincias inesperado'
    elif nombre == 'departamentos':
        gdf['id'] = gdf['in1'].astype(str).str.strip().apply(lambda x: 100000 + int(x))
    elif nombre == 'municipios':
        gdf['in1'] = gdf['in1'].astype(str).str.strip()
        ok = gdf['in1'].str.replace('.', '', regex=False).str.isdigit()
        assert ok.all(), f'{int((~ok).sum())} municipios sin in1'
        gdf['id'] = gdf['in1'].apply(lambda x: 1000000 + int(x))
    elif nombre == 'barrios':
        # Join al catalogo por nombre (nunca por posicion: el orden del Sheet
        # y del GPKG no coinciden). Requiere CAT_JURIS en la sesion.
        import pandas as pd, unicodedata
        assert CAT_JURIS.exists(), 'Subi data/registry/jurisdicciones.parquet a /content (drag & drop)'
        cat = pd.read_parquet(CAT_JURIS)
        cat = cat[cat['nivel'] == 'barrio'][['yvy_id', 'nombre_oficial']].copy()
        norm = lambda s: unicodedata.normalize('NFD', str(s)).encode('ascii', 'ignore').decode().lower().strip()
        cat['_k'] = cat['nombre_oficial'].apply(norm)
        dup_cat = cat[cat['_k'].duplicated(keep=False)]
        assert len(dup_cat) == 0, f'nombres duplicados en catalogo: {dup_cat["nombre_oficial"].tolist()}'
        cmap = dict(zip(cat['_k'], cat['yvy_id'].astype(int)))
        gdf['_k'] = gdf['nom_largo'].apply(norm)
        gdf['id'] = gdf['_k'].map(cmap)
        sin = gdf[gdf['id'].isna()]
        assert len(sin) == 0, f'{len(sin)} barrios sin match: {sin["nom_largo"].tolist()[:10]}'
        sin_geom = sorted(set(cat['yvy_id'].astype(int)) - set(gdf['id'].astype(int)))
        print(f'    barrios: {len(gdf)} feats, {len(sin_geom)} del catalogo sin geometria: {sin_geom}')
        gdf = gdf.drop(columns=['_k'])
    else:  # mineria, catastro: indice interno (celda_8)
        gdf = gdf.reset_index(drop=True)
        gdf['id'] = range(len(gdf))
    gdf['id'] = gdf['id'].astype('int64')
    assert not gdf['id'].isna().any(), f'{nombre}: ids nulos'
    assert not gdf['id'].duplicated().any(), f'{nombre}: ids duplicados'
    return gdf

def gpkg_a_pmtiles(gpkg, capa, salida, zmin, zmax):
    print(f'--- {salida.name}: leyendo {capa}...', flush=True)
    cols = ['in1', 'geom_estado'] if salida.stem in ('provincias', 'departamentos', 'municipios') \
        else (['nom_largo', 'geom_estado'] if salida.stem == 'barrios' else ['geom_estado'])
    gdf = gpd.read_file(gpkg, layer=capa, columns=cols) if cols \
        else gpd.read_file(gpkg, layer=capa)
    print(f'    {len(gdf)} feats en memoria', flush=True)
    if 'geom_estado' in gdf.columns:
        n0 = len(gdf)
        gdf = gdf[gdf['geom_estado'].isin(['ok', 'reparada'])].copy()
        print(f'    filtro geom_estado: {n0} -> {len(gdf)}', flush=True)
    gdf = asignar_id(gdf, capa, salida.stem)
    if str(gdf.crs) != 'EPSG:4326':
        print('    reproyectando a 4326...', flush=True)
        gdf = gdf.to_crs('EPSG:4326')
    gdf = gdf[['geometry', 'id']].copy()
    tmp = salida.parent / (salida.name + '.tmp.geojson')
    gdf.to_file(tmp, driver='GeoJSON')
    print(f'    geojson {tmp.stat().st_size/1e6:.1f} MB, tileando z{zmin}-{zmax}...', flush=True)
    tmp_pm = salida.parent / (salida.name + '.tmp')
    cmd = ['tippecanoe', '-o', str(tmp_pm), '-Z', str(zmin), '-z', str(zmax),
           '-l', salida.stem, '--use-attribute-for-id=id',
           '--maximum-tile-bytes=500000', '--force', str(tmp)]
    r = subprocess.run(cmd, text=True)  # log en vivo, sin capture
    tmp.unlink(missing_ok=True)
    assert r.returncode == 0, f'tippecanoe fallo (rc={r.returncode})'
    assert tmp_pm.read_bytes()[:7] == b'PMTiles', 'magic invalido'
    tmp_pm.replace(salida)  # atomico
    print(f'OK {salida.name} ({salida.stat().st_size/1e6:.1f} MB)')

for gpkg, capa, sal, zmin, zmax in PLAN:
    p = WORK / gpkg
    if not p.exists():
        print(f'SKIP {sal} (falta {gpkg})')
        continue
    gpkg_a_pmtiles(p, capa, OUT / sal, zmin, zmax)

In [ ]:
# 4) Verificacion: magic + header (min/max zoom) por archivo
import struct
for sal in sorted(OUT.glob('*.pmtiles')):
    h = sal.read_bytes()[:127]
    ok = h[:7] == b'PMTiles' and h[7] == 3
    zmin, zmax = h[100], h[101]
    print(f"{'OK' if ok else 'MAL'} {sal.name:24s} {sal.stat().st_size/1e6:7.1f} MB  z{zmin}-{zmax}")
print('\nSiguiente: copiar 08_pmtiles/*.pmtiles -> repo public/data/ + pnpm run build')